# Reproduce NA2Q on one cooperative MARL task

**Paper:** [NA2Q: Neural Attention Additive Model for Interpretable Multi-Agent Q-Learning](https://proceedings.mlr.press/v202/liu23be.html) (ICML 2023) <cite data-cite="pmlr-v202-liu23be">(Liu et al., 2023)</cite>  
**Reference code:** [zichuan-liu/NA2Q](https://github.com/zichuan-liu/NA2Q/tree/adc10b43bd2677179cb3e0f09bebd703e9debf9f) at `adc10b43bd2677179cb3e0f09bebd703e9debf9f`  
**Target:** Level-Based Foraging `lbf-4-2` (4 agents, 2 food, 10x10 grid, 5x5 local view)  
**Default execution:** bounded deterministic smoke fixture, not a scientific reproduction  
**Current evidence:** additive-decomposition mechanics, identity/mask semantics, reference-form fixture parity, and XDRL provenance only.

The public reference repository does not ship a checkpoint, frozen evaluation batch, or the paper's five-seed result data; its README asks readers to contact the authors for experimental data. Scientific mode therefore fails closed unless `XDRL_NA2Q_MODE=paper`, `XDRL_NA2Q_BUNDLE` points to an externally obtained bundle, and `XDRL_NA2Q_BUNDLE_SHA256` matches it. No later checkpoint, retrained policy, or compatible LBF build is silently substituted.


## Method and scope

The paper decomposes the joint value into unary and pairwise shape functions, weights them with state/identity attention, and adds a state bias (Eq. 7-8). This notebook keeps four separate layers of evidence:

1. **Adapter parity:** a saved deterministic batch must match independently frozen raw shapes, attention weights, and joint value.
2. **Structural invariants:** coalition membership, local masks, monotonicity, additive reconstruction, and individual-global-max policy consistency must pass.
3. **Task metrics:** average test return is reported with a 95% interval across five frozen smoke seeds, alongside VDN-like and random controls.
4. **Interpretability diagnostics:** local-mask IoU and top-coalition support precision are reported with shuffled-mask and uniform-attention controls. These synthetic diagnostics do not validate human-like interpretability.

Paper mode additionally requires the exact reference revision, the paper's LBF configuration (50-step episodes, batch size 32, test interval 10,000, 32 test episodes, replay size 5,000, discount 0.99, 1,050,000 steps, epsilon 1.0 to 0.05 over 50,000 steps, target update 200), five declared training seeds, an immutable environment identity, checkpoint and batch digests, and reference-exported tensors. The current paper path audits exported tensor mappings rather than independently running the checkpoint, so it cannot support the paper's scientific claims. One LBF scenario cannot establish all-task superiority, causal credit, or human-like interpretation.


In [ ]:
import itertools
import json
import os
import subprocess
from pathlib import Path

import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow
from torchrl.data import UnboundedContinuous
from xdrl import (
    AgentSelector,
    ArtifactDigestAlgorithm,
    BatchSemantics,
    bytes_digest,
    CoalitionTerm,
    InputArtifactReference,
    InputArtifactRole,
    InteractionContract,
    InteractionPhase,
    InteractionTopology,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    module_digest,
    MultiAgentSemantics,
    NamedReduction,
    OutputArtifactDeclaration,
    OutputArtifactDigest,
    OutputArtifactRole,
    RuntimeInteractionContext,
    SemanticTarget,
    TDHookWorkflowRunner,
    TensorDictSchema,
    tensor_digest,
    ValueDecompositionAxes,
    ValueDecompositionKeys,
    ValueDecompositionSemantics,
)

REFERENCE_REVISION = "adc10b43bd2677179cb3e0f09bebd703e9debf9f"
PAPER_SHA256 = "deb8aefc1e57d33fe6beabb909e696ec94459dbfebe9bde5e4755c049a3c8a03"
MODE = os.environ.get("XDRL_NA2Q_MODE", "smoke")
SEEDS = (1, 7, 19, 31, 43)
AGENTS = ("agent-0", "agent-1", "agent-2", "agent-3")
COALITIONS = tuple((i,) for i in range(4)) + tuple(itertools.combinations(range(4), 2))
TERM_NAMES = tuple("+".join(AGENTS[i] for i in members) for members in COALITIONS)
PAPER_CONFIG = {
    "environment": "lbf-4-2",
    "field_size": 10,
    "players": 4,
    "food": 2,
    "sight": 2,
    "max_episode_length": 50,
    "batch_size": 32,
    "test_interval": 10_000,
    "test_episodes": 32,
    "replay_batch_size": 5_000,
    "discount": 0.99,
    "total_timesteps": 1_050_000,
    "epsilon_start": 1.0,
    "epsilon_finish": 0.05,
    "epsilon_anneal_steps": 50_000,
    "target_update_interval": 200,
    "mix_nary": [1, 2],
    "rnn_hidden_dim": 64,
}
assert MODE in {"smoke", "paper"}
torch.manual_seed(SEEDS[0])

## Load exact external evidence or build the bounded smoke fixture

The paper bundle is a tensor-only export from the pinned reference code. It must preserve individual values, identity vectors, 5x5 local masks, every unary/pair shape output, attention weight, bias, joint value, chosen actions, five-seed returns, baseline returns, and interpretability diagnostics. The notebook never downloads or trains a replacement checkpoint.


In [ ]:
def evaluation_batch_digest(bundle_value):
    fields = ["agent_q", "state", "identity_semantics", "local_semantic_mask", "coalition_mask"]
    if "actions" in bundle_value:
        fields.append("actions")
    manifest = {name: tensor_digest(bundle_value[name]) for name in fields}
    return bytes_digest(json.dumps(manifest, sort_keys=True).encode())


def coalition_mask(batch_size):
    mask = torch.zeros(len(COALITIONS), len(AGENTS))
    for coalition_index, members in enumerate(COALITIONS):
        mask[coalition_index, list(members)] = 1
    return mask.expand(batch_size, -1, -1).clone()


def smoke_bundle(batch_size=32):
    generator = torch.Generator().manual_seed(5701)
    agent_q = 0.15 + 1.1 * torch.rand(batch_size, len(AGENTS), 1, generator=generator)
    agent_q[0, :, 0] = torch.tensor([0.2, 0.5, 0.7, 1.0])
    state = torch.randn(batch_size, 4, generator=generator) * 0.25
    state[0] = torch.tensor([0.4, -0.2, 0.1, 0.3])
    identity = torch.eye(len(AGENTS)).expand(batch_size, -1, -1).clone()
    local_masks = torch.zeros(batch_size, len(AGENTS), 5, 5)
    relevant_cells = ((1, 2), (2, 1), (2, 3), (3, 2))
    for agent, (row, column) in enumerate(relevant_cells):
        local_masks[:, agent, row, column] = 1
        local_masks[:, agent, 2, 2] = 1
    return {
        "agent_q": agent_q,
        "state": state,
        "identity_semantics": identity,
        "local_semantic_mask": local_masks,
        "coalition_mask": coalition_mask(batch_size),
        "reference_raw_shape_first": torch.tensor([0.21, 0.33, 0.41, 0.53, 0.265, 0.319, 0.4, 0.415, 0.505, 0.575]),
        "reference_attention_first": torch.tensor(
            [
                0.0691854226,
                0.0764617170,
                0.0845032660,
                0.0933905521,
                0.0915412952,
                0.1011687773,
                0.1118087905,
                0.1118087905,
                0.1235678236,
                0.1365635651,
            ]
        ),
        "reference_joint_first": torch.tensor(0.4124858854),
        "reference_bias": torch.zeros(batch_size, 1),
        "environment": "synthetic-lbf-shaped-smoke-v1",
        "config": {"episodes_per_seed": 32, "episode_length": 12, "action_count": 6},
    }


if MODE == "paper":
    bundle_path = Path(os.environ["XDRL_NA2Q_BUNDLE"])
    expected_sha = os.environ["XDRL_NA2Q_BUNDLE_SHA256"].lower()
    code_revision = os.environ["XDRL_CODE_REVISION"]
    running_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if code_revision != running_revision or subprocess.check_output(["git", "status", "--porcelain"], text=True):
        raise RuntimeError("paper mode requires the declared XDRL revision and a clean working tree")
    actual_bundle_sha = bytes_digest(bundle_path.read_bytes())
    if actual_bundle_sha != expected_sha:
        raise RuntimeError(f"paper bundle digest mismatch: expected {expected_sha}, got {actual_bundle_sha}")
    bundle = torch.load(bundle_path, map_location="cpu", weights_only=True)
    required = {
        "agent_q",
        "state",
        "identity_semantics",
        "local_semantic_mask",
        "coalition_mask",
        "reference_raw_shape",
        "reference_attention",
        "reference_bias",
        "reference_joint_value",
        "actions",
        "episode_returns",
        "baseline_returns",
        "interpretability_metrics",
        "checkpoint_sha256",
        "evaluation_batch_sha256",
        "reference_revision",
        "environment_build",
        "config",
        "seeds",
    }
    if set(bundle) != required:
        raise RuntimeError(f"paper bundle schema mismatch: {sorted(set(bundle) ^ required)}")
    if bundle["reference_revision"] != REFERENCE_REVISION or bundle["config"] != PAPER_CONFIG:
        raise RuntimeError("paper bundle does not match the pinned NA2Q revision and LBF protocol")
    if tuple(bundle["seeds"]) != SEEDS or not bundle["environment_build"].get("immutable", False):
        raise RuntimeError("paper bundle must pin five declared seeds and an immutable environment build")
    if evaluation_batch_digest(bundle) != bundle["evaluation_batch_sha256"]:
        raise RuntimeError("paper evaluation batch digest mismatch")
else:
    bundle = smoke_bundle()
    actual_bundle_sha = bytes_digest(b"na2q-synthetic-lbf-shaped-smoke-v1")
    code_revision = "working-tree-smoke"

{
    "mode": MODE,
    "environment": bundle["environment"] if MODE == "smoke" else bundle["environment_build"],
    "bundle_sha256": actual_bundle_sha,
    "reference_revision": REFERENCE_REVISION,
    "claim_ready": False,
}

## Map PyMARL values into TensorDict semantics

| NA2Q/PyMARL value | TensorDict key | Ordered axes | Role |
|---|---|---|---|
| chosen local `Q_i` | `agents/individual_value` | `episode, agent, value_feature` | individual value |
| unary/pair membership | `decomposition/coalition_mask` | `episode, coalition, agent` | semantic mask |
| global state plus identity semantics | `mixer/context` | `episode, context_feature` | mixer input |
| raw `f_i`, `f_ij` | `decomposition/raw_shape` | `episode, coalition, value_feature` | preserved feature |
| attention `alpha_i`, `alpha_ij` | `decomposition/attention` | `episode, coalition, value_feature` | preserved feature |
| state bias `f0(s)` | `mixer/state_bias` | `episode, value_feature` | preserved feature |
| reconstructive terms `alpha_k f_k + f0/10` | `decomposition/contribution` | `episode, coalition, value_feature` | coalition contribution |
| `Q_tot` | `mixer/joint_value` | `episode, value_feature` | joint value |
| local identity mask | `agents/local_semantic_mask` | `episode, agent, local_y, local_x` | preserved feature |

The one mixer module path is execution structure, not an agent or coalition identity. Semantic identity is carried only by the ordered agent list, `CoalitionTerm` membership, masks, and named axes. Raw shape functions, attention weights, and the unallocated state bias remain separate. Because the current XDRL value-decomposition contract has no separate bias edge, `coalition_contribution` uses a declared uniform `f0/10` allocation solely to make the named reduction exact; those reconstructive terms must not be interpreted as coalition credit.


In [ ]:
KEYS = ValueDecompositionKeys(
    individual_value=("agents", "individual_value"),
    coalition_contribution=("decomposition", "contribution"),
    semantic_mask=("decomposition", "coalition_mask"),
    mixer_input=("mixer", "context"),
    joint_value=("mixer", "joint_value"),
)
RAW_KEY = ("decomposition", "raw_shape")
ATTENTION_KEY = ("decomposition", "attention")
BIAS_KEY = ("mixer", "state_bias")
IDENTITY_KEY = ("agents", "identity_semantics")
LOCAL_MASK_KEY = ("agents", "local_semantic_mask")
terms = tuple(
    CoalitionTerm(TERM_NAMES[index], tuple(AGENTS[i] for i in members), index)
    for index, members in enumerate(COALITIONS)
)
axes = ValueDecompositionAxes(
    individual_value=("episode", "agent", "value_feature"),
    coalition_contribution=("episode", "coalition", "value_feature"),
    semantic_mask=("episode", "coalition", "agent"),
    mixer_input=("episode", "context_feature"),
    joint_value=("episode", "value_feature"),
)
reduction = NamedReduction(
    "sum_reconstructive_terms_after_uniform_bias_allocation",
    KEYS.coalition_contribution,
    KEYS.joint_value,
    ("coalition",),
)
batch_semantics = BatchSemantics(("episode",))
context_size = (
    bundle["state"].shape[-1] + bundle["identity_semantics"].shape[-2] * bundle["identity_semantics"].shape[-1]
)
inputs = TensorDictSchema(
    (
        KeySchema(
            KEYS.individual_value, KeyRole.INDIVIDUAL_VALUE, KeyPresence.REQUIRED, UnboundedContinuous(shape=(4, 1))
        ),
        KeySchema(KEYS.semantic_mask, KeyRole.SEMANTIC_MASK, KeyPresence.REQUIRED, UnboundedContinuous(shape=(10, 4))),
        KeySchema(
            KEYS.mixer_input, KeyRole.MIXER_INPUT, KeyPresence.REQUIRED, UnboundedContinuous(shape=(context_size,))
        ),
        KeySchema(IDENTITY_KEY, KeyRole.FEATURE, KeyPresence.REQUIRED),
        KeySchema(LOCAL_MASK_KEY, KeyRole.FEATURE, KeyPresence.REQUIRED, UnboundedContinuous(shape=(4, 5, 5))),
    ),
    batch_semantics,
)
outputs = TensorDictSchema(
    (
        KeySchema(RAW_KEY, KeyRole.FEATURE, KeyPresence.PRODUCED, UnboundedContinuous(shape=(10, 1))),
        KeySchema(ATTENTION_KEY, KeyRole.FEATURE, KeyPresence.PRODUCED, UnboundedContinuous(shape=(10, 1))),
        KeySchema(BIAS_KEY, KeyRole.FEATURE, KeyPresence.PRODUCED, UnboundedContinuous(shape=(1,))),
        KeySchema(
            KEYS.coalition_contribution,
            KeyRole.COALITION_CONTRIBUTION,
            KeyPresence.PRODUCED,
            UnboundedContinuous(shape=(10, 1)),
        ),
        KeySchema(KEYS.joint_value, KeyRole.JOINT_VALUE, KeyPresence.PRODUCED, UnboundedContinuous(shape=(1,))),
    ),
    batch_semantics,
)
semantics = ValueDecompositionSemantics(
    coalition_axis="coalition",
    feature_axes=("value_feature", "context_feature", "local_y", "local_x"),
    terms=terms,
    keys=KEYS,
    axes=axes,
    reductions=(reduction,),
    parameters_shared=MODE == "smoke",
)
multi_agent = MultiAgentSemantics(
    InteractionTopology.MIXER,
    "agents",
    len(AGENTS),
    SemanticTarget(ModelRole.MIXER, AgentSelector("agents")),
    agent_identities=AGENTS,
)
contract = InteractionContract(
    identity="na2q:lbf-4-2:value-decomposition",
    role=ModelRole.MIXER,
    phase=InteractionPhase.EVALUATION,
    module_path="learner.mixer",
    input_schema=inputs,
    output_schema=outputs,
    environment=PAPER_CONFIG["environment"] if MODE == "paper" else bundle["environment"],
    agent_dimension="agent",
    model_id="NA2Q-QNAMer" if MODE == "paper" else "deterministic-reference-form-smoke",
    checkpoint_id=(f"sha256:{bundle['checkpoint_sha256']}" if MODE == "paper" else None),
    module_training=False,
    multi_agent=multi_agent,
    value_decomposition=semantics,
)
assert contract.module_path not in TERM_NAMES
assert tuple(term.identity for term in semantics.terms) == TERM_NAMES

## Deterministic additive adapter and saved-batch parity

The smoke mixer is deliberately small: one shared monotone unary map, one shared monotone pair map, and deterministic state/identity attention that consumes the declared identity tensor. It mirrors Eq. 7-8 but is not the authors' trained network. In paper mode, the adapter consumes reference-exported raw shapes and attention, so the notebook checks tensor mapping and additive reconstruction without copying unlicensed reference implementation code. Because that path does not independently execute the checkpoint, it cannot establish reference agreement or claim readiness.


In [ ]:
class NA2QDecompositionAdapter(torch.nn.Module):
    def __init__(self, recorded=None):
        super().__init__()
        self.recorded = recorded

    def forward(self, individual_value, coalition_mask_value, context, identity_semantics):
        if self.recorded is not None:
            raw = self.recorded["reference_raw_shape"].reshape(len(individual_value), len(COALITIONS), 1)
            attention = self.recorded["reference_attention"].reshape(len(individual_value), len(COALITIONS), 1)
            bias = self.recorded["reference_bias"].reshape(len(individual_value), 1)
        else:
            selected = individual_value.squeeze(-1).unsqueeze(1) * coalition_mask_value
            sizes = coalition_mask_value.sum(-1)
            total = selected.sum(-1)
            product = torch.where(
                sizes == 2,
                torch.where(coalition_mask_value.bool(), individual_value.squeeze(-1).unsqueeze(1), 1).prod(-1),
                torch.zeros_like(total),
            )
            raw = torch.where(sizes == 1, 0.4 * total + 0.13, 0.25 * total + 0.1 * product + 0.08).unsqueeze(-1)
            state_signal = context[:, :1]
            identity_weights = torch.arange(
                1, identity_semantics.shape[-1] + 1, dtype=context.dtype, device=context.device
            )
            identity_codes = torch.einsum("eaf,f->ea", identity_semantics, identity_weights)
            scores = 0.2 * state_signal * sizes + 0.1 * (coalition_mask_value * identity_codes.unsqueeze(1)).sum(-1)
            attention = scores.softmax(-1).unsqueeze(-1)
            bias = torch.zeros(len(individual_value), 1, dtype=context.dtype, device=context.device)
        contribution = raw * attention + bias.unsqueeze(-2) / len(COALITIONS)
        joint = contribution.sum(dim=-2)
        return raw, attention, bias, contribution, joint


context = torch.cat((bundle["state"], bundle["identity_semantics"].flatten(1)), dim=-1).float()
data = TensorDict(
    {
        KEYS.individual_value: bundle["agent_q"].float(),
        KEYS.semantic_mask: bundle["coalition_mask"].float(),
        KEYS.mixer_input: context,
        IDENTITY_KEY: bundle["identity_semantics"].float(),
        LOCAL_MASK_KEY: bundle["local_semantic_mask"].float(),
    },
    batch_size=[len(bundle["agent_q"])],
)
adapter = NA2QDecompositionAdapter(bundle if MODE == "paper" else None)
mixer = TensorDictModule(
    adapter,
    in_keys=[KEYS.individual_value, KEYS.semantic_mask, KEYS.mixer_input, IDENTITY_KEY],
    out_keys=[RAW_KEY, ATTENTION_KEY, BIAS_KEY, KEYS.coalition_contribution, KEYS.joint_value],
)
native = RuntimeInteractionContext(contract, mixer, data)(data.clone())

reference_export_reconstruction = None
if MODE == "smoke":
    torch.testing.assert_close(native[RAW_KEY][0, :, 0], bundle["reference_raw_shape_first"], rtol=0, atol=1e-7)
    torch.testing.assert_close(native[ATTENTION_KEY][0, :, 0], bundle["reference_attention_first"], rtol=0, atol=1e-7)
    torch.testing.assert_close(native[KEYS.joint_value][0, 0], bundle["reference_joint_first"], rtol=0, atol=1e-7)
    smoke_saved_batch_parity = True
else:
    smoke_saved_batch_parity = None
    reference_export_reconstruction = bool(
        torch.allclose(native[KEYS.joint_value], bundle["reference_joint_value"], rtol=1e-6, atol=1e-7)
    )

{
    "smoke_saved_batch_parity": smoke_saved_batch_parity,
    "reference_export_reconstruction": reference_export_reconstruction,
    "individual_value_shape": tuple(data[KEYS.individual_value].shape),
    "raw_shape_shape": tuple(native[RAW_KEY].shape),
    "attention_shape": tuple(native[ATTENTION_KEY].shape),
    "state_bias_shape": tuple(native[BIAS_KEY].shape),
    "coalition_contribution_shape": tuple(native[KEYS.coalition_contribution].shape),
    "joint_value_shape": tuple(native[KEYS.joint_value].shape),
}

## Execute through XDRL and retain typed artifact evidence

The workflow caches coalition outputs without collapsing the coalition axis. Input artifacts distinguish the model/checkpoint, frozen evaluation batch, and paper PDF. The result declaration resolves only after successful execution. In smoke mode, synthetic identities are explicitly labeled and are not scientific assets.


In [ ]:
checkpoint_sha = bundle["checkpoint_sha256"] if MODE == "paper" else module_digest(adapter)
batch_sha = bundle["evaluation_batch_sha256"] if MODE == "paper" else evaluation_batch_digest(bundle)
input_artifacts = (
    InputArtifactReference(
        "checkpoint:na2q-lbf-4-2" if MODE == "paper" else "checkpoint:na2q-smoke-adapter",
        InputArtifactRole.MODEL_CHECKPOINT,
        ArtifactDigestAlgorithm.SHA256,
        checkpoint_sha,
        source="https://github.com/zichuan-liu/NA2Q",
        revision=REFERENCE_REVISION,
        metadata={"mode": MODE, "paper_checkpoint": MODE == "paper"},
    ),
    InputArtifactReference(
        "batch:na2q-lbf-4-2-evaluation" if MODE == "paper" else "batch:na2q-smoke-fixture",
        InputArtifactRole.EVALUATION_SPLIT,
        ArtifactDigestAlgorithm.SHA256,
        batch_sha,
        metadata={"mode": MODE, "examples": len(data), "frozen": True},
    ),
    InputArtifactReference(
        "paper:liu23be",
        InputArtifactRole.OTHER,
        ArtifactDigestAlgorithm.SHA256,
        PAPER_SHA256,
        source="https://proceedings.mlr.press/v202/liu23be/liu23be.pdf",
        revision="PMLR-v202",
        source_is_immutable=True,
        metadata={"used_for": "equations, protocol, and declared paper comparisons; not numeric curve extraction"},
    ),
    InputArtifactReference(
        "reference:na2q-export-bundle" if MODE == "paper" else "reference:na2q-smoke-fixture",
        InputArtifactRole.REFERENCE_RESULT,
        ArtifactDigestAlgorithm.SHA256,
        actual_bundle_sha,
        source="https://github.com/zichuan-liu/NA2Q",
        revision=REFERENCE_REVISION,
        metadata={"mode": MODE, "contains_reference_exported_tensors": MODE == "paper"},
    ),
)


def cached_contributions(output, **_):
    return output[3]


def result_resolver(result, declarations):
    digest = tensor_digest(result[KEYS.joint_value])
    return tuple(OutputArtifactDigest(item.identity, ArtifactDigestAlgorithm.SHA256, digest) for item in declarations)


runner = TDHookWorkflowRunner(RuntimeInteractionContext(contract, mixer, data))
execution = runner.run(
    Workflow(ActivationCaching("module", callback=cached_contributions)),
    data.clone(),
    code_revision=code_revision,
    seed=SEEDS[0],
    input_artifacts=input_artifacts,
    output_artifacts=(
        OutputArtifactDeclaration("result:na2q-joint-value", OutputArtifactRole.ATTRIBUTION, metadata={"mode": MODE}),
    ),
    output_artifact_resolver=result_resolver,
    callback_identifiers={cached_contributions: "coalition-contribution-v1"},
)
torch.testing.assert_close(execution.data[KEYS.joint_value], native[KEYS.joint_value], rtol=0, atol=0)
assert execution.provenance.input_artifacts == input_artifacts
assert len(execution.provenance.output_artifacts) == 1
{
    "instrumented_parity": True,
    "model_calls": execution.provenance.model_calls,
    "coalitions_preserved": execution.data[KEYS.coalition_contribution].shape[-2] == len(COALITIONS),
    "output_artifact": execution.provenance.output_artifacts[0].to_dict(),
}

## Verify decomposition and induced-policy invariants

Attention is normalized across named coalitions, masked-out agents cannot affect a coalition term, and `Q_tot` is reduced only after raw shapes and weights are retained. For the non-negative smoke maps, increasing any individual value cannot lower the joint value. Exhaustive one-step action values verify that independent greedy choices recover the joint argmax (the IGM condition) for the fixture.


In [ ]:
result = execution.data
weighted_reconstruction = (result[RAW_KEY] * result[ATTENTION_KEY]).sum(-2) + result[BIAS_KEY]
expected_mask = coalition_mask(len(result))
additive_reconstruction = bool(
    torch.allclose(result[KEYS.joint_value], weighted_reconstruction, rtol=1e-6, atol=1e-7)
    and torch.allclose(result[KEYS.joint_value], result[KEYS.coalition_contribution].sum(-2), rtol=1e-6, atol=1e-7)
)
attention_normalized = bool(
    torch.allclose(result[ATTENTION_KEY].sum(-2), torch.ones_like(result[KEYS.joint_value]), rtol=1e-6, atol=1e-7)
)
membership_mask_exact = bool(torch.equal(result[KEYS.semantic_mask], expected_mask))

policy_parity = None
monotone = None
if MODE == "smoke":
    perturbed = data.clone()
    perturbed[KEYS.individual_value] = perturbed[KEYS.individual_value] + 0.05
    perturbed_joint = mixer(perturbed)[KEYS.joint_value]
    monotone = bool((perturbed_joint >= result[KEYS.joint_value] - 1e-7).all())
    action_values = torch.tensor(
        [[0.2, 0.5, 0.1], [0.4, 0.3, 0.6], [0.7, 0.2, 0.1], [0.1, 0.8, 0.4]], dtype=torch.float
    )
    joint_scores = []
    actions = tuple(itertools.product(range(action_values.shape[-1]), repeat=len(AGENTS)))
    for joint_action in actions:
        candidate = data[:1].clone()
        candidate[KEYS.individual_value] = torch.stack(
            [action_values[agent, action] for agent, action in enumerate(joint_action)]
        ).reshape(1, len(AGENTS), 1)
        joint_scores.append(float(mixer(candidate)[KEYS.joint_value][0, 0]))
    independent_greedy = tuple(action_values.argmax(-1).tolist())
    policy_parity = actions[int(torch.tensor(joint_scores).argmax())] == independent_greedy
else:
    policy_parity = bool(bundle["interpretability_metrics"].get("igm_policy_parity", False))
    monotone = bool(bundle["interpretability_metrics"].get("monotonicity", False))

invariants = {
    "additive_reconstruction": additive_reconstruction,
    "attention_normalized": attention_normalized,
    "membership_mask_exact": membership_mask_exact,
    "monotonicity": monotone,
    "induced_policy_parity": policy_parity,
    "module_path_is_not_semantic_identity": contract.module_path not in TERM_NAMES,
    "per_agent_and_coalition_outputs_preserved": True,
}
if MODE == "smoke":
    assert all(invariants.values())
invariants

## Task and interpretability metrics with frozen controls

The smoke values below are generated from a known synthetic signal so CI/reporting and negative-control plumbing execute quickly. They are not LBF rollouts and are never compared numerically with Figure 3. Paper mode reads complete per-seed/per-episode arrays from the checksummed result bundle; it does not digitize plots.


In [ ]:
def bootstrap_mean_ci(independent_values):
    values = torch.as_tensor(independent_values, dtype=torch.float)
    if values.ndim != 1 or not len(values) or not torch.isfinite(values).all():
        raise ValueError("confidence intervals require a finite one-dimensional independent-sample tensor")
    generator = torch.Generator().manual_seed(991)
    draws = torch.randint(len(values), (2000, len(values)), generator=generator)
    bootstrap = values[draws].mean(-1)
    lower, upper = torch.quantile(bootstrap, torch.tensor([0.025, 0.975]))
    return {"mean": float(values.mean()), "lower_95": float(lower), "upper_95": float(upper), "n": len(values)}


def task_return_ci(values):
    values = torch.as_tensor(values, dtype=torch.float)
    expected = (len(SEEDS), PAPER_CONFIG["test_episodes"])
    if tuple(values.shape) != expected:
        raise ValueError(f"task returns must have shape {expected}, got {tuple(values.shape)}")
    return bootstrap_mean_ci(values.mean(dim=-1))


def diagnostic_ci(values):
    values = torch.as_tensor(values, dtype=torch.float)
    expected = (len(SEEDS),)
    if tuple(values.shape) != expected:
        raise ValueError(f"interpretability diagnostics must have shape {expected}, got {tuple(values.shape)}")
    return bootstrap_mean_ci(values)


if MODE == "smoke":
    generator = torch.Generator().manual_seed(5702)
    seed_offset = torch.tensor(SEEDS, dtype=torch.float).unsqueeze(1) * 0.0004
    noise = torch.randn(len(SEEDS), 32, generator=generator) * 0.025
    episode_returns = {
        "na2q_smoke": (0.78 + seed_offset + noise).clamp(0, 1),
        "vdn_like_control": (0.68 + seed_offset + noise * 1.15).clamp(0, 1),
        "random_policy_control": (0.18 + noise * 1.4).clamp(0, 1),
    }
    mask_iou = torch.tensor([1.0, 0.90, 0.95, 0.92, 0.97])
    shuffled_mask_iou = torch.tensor([0.12, 0.18, 0.10, 0.15, 0.20])
    coalition_precision = torch.tensor([0.90, 0.85, 0.95, 0.88, 0.92])
    uniform_attention_precision = torch.tensor([0.40, 0.35, 0.45, 0.40, 0.38])
else:
    if not isinstance(bundle["episode_returns"], dict) or set(bundle["episode_returns"]) != {"na2q"}:
        raise ValueError("paper episode_returns must be a mapping with exactly the 'na2q' method")
    if not isinstance(bundle["baseline_returns"], dict) or not bundle["baseline_returns"]:
        raise ValueError("paper baseline_returns must be a non-empty method mapping")
    if set(bundle["episode_returns"]) & set(bundle["baseline_returns"]):
        raise ValueError("paper task metric method names must be unique")
    episode_returns = bundle["episode_returns"] | bundle["baseline_returns"]
    metrics = bundle["interpretability_metrics"]
    required_metrics = {
        "local_mask_iou",
        "shuffled_mask_iou",
        "coalition_support_precision",
        "uniform_attention_support_precision",
        "igm_policy_parity",
        "monotonicity",
    }
    if not isinstance(metrics, dict) or set(metrics) != required_metrics:
        raise ValueError("paper interpretability_metrics has an unknown schema")
    mask_iou = metrics["local_mask_iou"]
    shuffled_mask_iou = metrics["shuffled_mask_iou"]
    coalition_precision = metrics["coalition_support_precision"]
    uniform_attention_precision = metrics["uniform_attention_support_precision"]

task_metrics = {name: task_return_ci(values) for name, values in episode_returns.items()}
interpretability_metrics = {
    "local_mask_iou": diagnostic_ci(mask_iou),
    "shuffled_mask_iou_control": diagnostic_ci(shuffled_mask_iou),
    "coalition_support_precision": diagnostic_ci(coalition_precision),
    "uniform_attention_control": diagnostic_ci(uniform_attention_precision),
}
{"task": task_metrics, "interpretability": interpretability_metrics}

## Results and scope

Execution, exact-asset availability, reference agreement, empirical metrics, and claim scope remain separate. A passing smoke fixture demonstrates maintenance of the adapter and evidence schema only. Reference differences are explicit: XDRL uses TensorDict keys and named reductions around exported tensors; it does not port the PyMARL replay/training stack, VAE, grouped-convolution shape networks, or environment runner into XDRL core.


In [ ]:
independent_reference_agreement = None
structural_invariants_passed = all(invariants.values())

results_summary = {
    "smoke_execution": "passed",
    "paper_assets": "verified" if MODE == "paper" else "not supplied",
    "reference_revision": REFERENCE_REVISION,
    "reference_agreement": "not independently evaluated (export-only adapter)" if MODE == "paper" else "not evaluated",
    "reference_export_reconstruction": ("passed" if reference_export_reconstruction else "failed")
    if MODE == "paper"
    else "not evaluated",
    "structural_invariants": "passed" if structural_invariants_passed else "failed",
    "task_metrics": "paper evaluation" if MODE == "paper" else "synthetic smoke only",
    "interpretability_metrics": "paper diagnostics" if MODE == "paper" else "synthetic known-signal diagnostics only",
    "claim_ready": False,
    "unsupported_reference_behavior": [
        "PyMARL training, replay, VAE learning, and environment execution remain external",
        "the public repository provides no paper checkpoint or five-seed result bundle",
        "the paper path maps exported tensors and does not independently execute the checkpoint",
        "XDRL provenance records tensor mapping and execution mechanics, not causal credit",
    ],
    "claim_limit": "One LBF scenario cannot establish all-task superiority, causal credit, or human-like interpretability.",
}
if MODE == "paper":
    output_root = Path(os.environ["XDRL_NA2Q_OUTPUT_ROOT"])
    output_root.mkdir(parents=True, exist_ok=True)
    evidence = {
        "task_metrics": task_metrics,
        "interpretability_metrics": interpretability_metrics,
        "invariants": invariants,
        "results_summary": results_summary,
        "decomposition_provenance": execution.provenance.to_dict(),
    }
    payload = json.dumps(evidence, indent=2, sort_keys=True) + "\n"
    output_path = output_root / "na2q-value-decomposition-evidence.json"
    output_path.write_text(payload)
    evidence_digest = bytes_digest(payload.encode())

    def evidence_resolver(_result, declarations):
        return tuple(
            OutputArtifactDigest(item.identity, ArtifactDigestAlgorithm.SHA256, evidence_digest)
            for item in declarations
        )

    evidence_execution = runner.run(
        Workflow(ActivationCaching("module", callback=cached_contributions)),
        data.clone(),
        code_revision=code_revision,
        seed=SEEDS[0],
        input_artifacts=input_artifacts,
        output_artifacts=(
            OutputArtifactDeclaration(
                "result:na2q-full-evidence",
                OutputArtifactRole.METRICS_BUNDLE,
                source=str(output_path),
                revision=evidence_digest,
                metadata={"retained_on_failed_agreement": True},
            ),
        ),
        output_artifact_resolver=evidence_resolver,
        callback_identifiers={cached_contributions: "coalition-contribution-v1"},
    )
    provenance_path = output_root / "na2q-value-decomposition-provenance.json"
    provenance_path.write_text(evidence_execution.provenance.to_json() + "\n")
    results_summary["full_result_artifact"] = {
        "path": str(output_path),
        "sha256": evidence_digest,
        "provenance_path": str(provenance_path),
        "typed_output_reference": evidence_execution.provenance.output_artifacts[0].to_dict(),
        "retained_on_failed_agreement": True,
    }
results_summary